In [ ]:
from datetime import timedelta

import joblib
import numpy as np
import pandas as pd

phase_model = joblib.load("cycle_phase_model.joblib")
hormone_models = joblib.load("hormone_models.joblib")

PHASES = phase_model.named_steps["clf"].classes_


def forecast_hormones(
    cycle_day,
    phase_probs,
):
    """
    cycle_day: int
    phase_probs: dict {phase: probability}
    """

    predictions = {}

    for hormone, model in hormone_models.items():
        hormone_estimate = 0.0

        for phase, prob in phase_probs.items():
            row = {
                "cycle_day": cycle_day,
                **{f"phase_{p}": 1 if p == phase else 0 for p in PHASES},
            }
            X = pd.DataFrame([row])
            hormone_estimate += prob * model.predict(X)[0]

        predictions[hormone] = hormone_estimate

    return predictions


def smooth_phase_probs(phase_probs, cycle_day):
    transitions = {
        "Menstrual": "Follicular",
        "Follicular": "Ovulation",
        "Ovulation": "Luteal",
        "Luteal": "Menstrual",
    }

    new_probs = phase_probs.copy()

    for phase, next_phase in transitions.items():
        new_probs[next_phase] += 0.1 * phase_probs[phase]
        new_probs[phase] *= 0.9

    total = sum(new_probs.values())
    return {k: v / total for k, v in new_probs.items()}


def forecast_cycle(start_cycle_day, cycle_length, days_ahead, initial_phase_probs):
    forecasts = []

    phase_probs = initial_phase_probs.copy()

    for day in range(days_ahead):
        cycle_day = (start_cycle_day + day) % cycle_length + 1

        hormones = forecast_hormones(cycle_day, phase_probs)

        forecasts.append({"cycle_day": cycle_day, **hormones, **phase_probs})

        # Optional: smooth phase transitions
        phase_probs = smooth_phase_probs(phase_probs, cycle_day)

    return forecasts


import pprint

# Load the models you trained
phase_model = joblib.load("cycle_phase_model.joblib")
hormone_models = joblib.load("hormone_models.joblib")

PHASES = phase_model.named_steps["clf"].classes_

# Import the functions from your forecasting code
# Make sure forecast_hormones and forecast_cycle are in scope
# If in another file, you can do:
# from forecast_utils import forecast_hormones, forecast_cycle, smooth_phase_probs

# -----------------------------
# 1. Set up fake phase probabilities for day 1
# -----------------------------
initial_phase_probs = {
    "Menstrual": 0.7,
    "Follicular": 0.2,
    "Ovulation": 0.05,
    "Luteal": 0.05,
}

# -----------------------------
# 2. Forecast 10 days ahead for a 28-day cycle
# -----------------------------
forecasted = forecast_cycle(
    start_cycle_day=1,
    cycle_length=28,
    days_ahead=128,
    initial_phase_probs=initial_phase_probs,
)

# -----------------------------
# 3. Print results nicely
# -----------------------------
pp = pprint.PrettyPrinter(indent=2)
for day in forecasted:
    print(f"Cycle Day {day['cycle_day']}:")
    pp.pprint(day)
    print("-" * 30)


: 